# Mixture of Experts (MoE)

A dense 70B transformer activates every parameter for every token. A 671B MoE activates only 37 per token and beats it on every benchmark. Sparsity is the most important scaling idea of the decade.

## Problem definition

A dense transformer's FLOPs at inference equal its parameter count (times 2 for forward pass). Scale up a dense model and every token pays the full bill. By 2024 the frontier was hitting a compute wall: to be meaningfully smarter, you needed exponentially more FLOPs per token.

Mixtrue of Experts breaks this link. Replace each FFM with E independent experts + a router that picks k experts per token. Total parameter = E x FFN_size. Active paramters per token = k x FFN_size. 

## Basic Concept

MoE layer： router selects k of E experts per token.

### The FFN swap

Dense transformer block:
```
h = x + attn(norm(x))
h = h + FFN(norm(h))
```

MoE block:
```
h = x + attn(norm(x))
scores = router(norm(h))
top_k = argmax_k(scores)
h = h + sum_{e in top_k}(
    gate(scores[e]) * Expert_e(norm(h))
)
```

Every expert is an independent FFM (typically SwiGLU). The router is a single linear layer. Each token picks its own k
experts and gets a gated mixture of their outputs.

### Load-balancing problem

if the router puts 90% of tokens through expert 3, the other expert starve. Three fixes have been tried:

1. Auxiliary load-balancing loss.  Add a penalty proportional to the variance in expert usage, Works, but adds a hyperparameter and a second gradient signal.
2. Expert capacity + token dropping.  Each expert processes at most C x N/E tokens overflow tokens skip the layer. Hurts quality.
3. Auxiliary-loss-free balancing. Add a learned per-expert bias that shifts the router's top-k selection. Bias is updated outside the training loss. No penalty on the main objective.

DeepSeek V3's approach: after each training step, for every expert, check if its usage is above or below the target. Nudge the bias by +-y. Selection uses scores + bias. Expert probabilities used for gating are the raw scores unchanged.

### Shared experts

Split experts into shared and routed. Every token passes through all shared experts, Routed experts are pick via top-k. Shared experts capture common knowledge.

### Fine-grained experts

Class MoE: each expert is as wide as a full FFN. E is small, k is small.

Modern fine-grained MoE: each expert is narrower (1/8 FFN size). E is large, k is large. Same total parameters, but combinations scale much faster.

# Build your Own

In [ ]:
import math
import random

import sys
from pathlib import Path

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

def silu(x):
    return x / (1.0 + math.exp(-x))

def make_expert(d_in, d_hidden, rng):
    scale = math.sqrt(2.0 / (d_in + d_hidden))
    W = [
        [rng.gauss(0.0, scale) for _ in range(d_hidden)]
        for _ in range(d_in)
    ]

    return W

def apply_expert(x, W):
    d_hidden = len(W[0])
    out = [0.0] * d_hidden
    for i, xi in enumerate(x):
        if xi == 0.0:
            continue
        for j in range(d_hidden):
            out[j] = xi * W[i][j]

    return [silu(v) for v in out]

def route(hidden, W_router, top_k, bias):
    E = len(W_router)
    scores = [
        sum(h * w for h, w in zip(hidden, W_router[e]))
        for e in range(E)
    ]

    biased = [
        s + b for s, b in zip(scores, bias)
    ]

    top_idx = sorted(range(E), key=lambda i:-biased[i])[:top_k]
    chosen = [scores[i] for i in top_idx]
    m = max(chosen)
    exp = [math.exp(c - m) for c in chosen]
    s = sum(exp)
    gates = [e / s for e in exp]

    return top_idx, gates

def moe_layer_forward(x, experts, W_router, top_k, bias):
    top_idx, gates = route(x, W_router, top_k, bias)
    d_hidden = len(experts[0][0])
    out = [0.0] * d_hidden
    for e_idx, gate in zip(top_idx, gates):
        h = apply_expert(x, experts[e_idx])
        for j in range(d_hidden):
            out[j] += gate * h[j]
    
    return out, top_idx

def update_bias(bias, usage_counts, target, gamma):
    for e in range(len(bias)):
        if usage_counts[e] > target:
            bias[e] -= gamma
        elif usage_counts[e] < target:
            bias[e] += gamma

    return bias

def run_epoch(tokens, experts, W_router, top_k, bias):
    usage = [0] * len(experts)
    for x in tokens:
        _, top_idx = moe_layer_forward(x, experts, W_router, top_k, bias)
        for e in top_idx:
            usage[e] += 1

    return usage

    